In [ ]:
# Enterprise FinTech Payment Intelligence Platform
# Phase 4 - Machine Learning
# Notebook 04 - Data Preprocessing

"""
Objective:
Prepare the feature-engineered dataset for machine learning by selecting
features, splitting the data, applying feature scaling, balancing the
training data using SMOTE, and exporting the final datasets for model training.
"""

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings("ignore")

In [3]:
# Load the feature-engineered dataset
df = pd.read_csv("feature_engineered_dataset.csv")

print("Dataset Loaded Successfully")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

df.head()

Dataset Loaded Successfully
Rows    : 6,362,620
Columns : 24


,TransactionID,DayNumber,HourOfSimulation,SourceAccountID,DestinationAccountID,IsFraud,IsFlaggedFraud,Amount,OldBalanceOrig,NewBalanceOrig,...,AmountToOriginBalanceRatio,HighValueTransaction,BalanceGap,PeriodOfDay_Evening,PeriodOfDay_Morning,PeriodOfDay_Night,TransactionType_CASH_OUT,TransactionType_DEBIT,TransactionType_PAYMENT,TransactionType_TRANSFER
0,20666,1,13,C2108058134,M1963634359,0,0,22005.03,0.00,0.00,...,2.200503e+04,0,0.00,0,0,0,0,0,1,0
1,20851,1,15,C1672847392,C882180306,0,0,214524.46,5030.00,219554.46,...,4.264052e+01,0,5030.00,0,0,0,0,0,0,0
2,21169,1,15,C519692057,C834458122,0,0,1432648.47,0.00,0.00,...,1.432648e+06,1,1453236.68,0,0,0,0,0,0,1
3,21181,1,15,C1634465843,C807329566,0,0,352807.10,0.00,0.00,...,3.528071e+05,0,426401.78,0,0,0,1,0,0,0
4,21321,1,15,C210125456,C410910993,0,0,275348.64,390959.31,115610.67,...,7.042880e-01,0,390959.31,0,0,0,1,0,0,0


In [4]:
print("=" * 60)
print("Dataset Information")
print("=" * 60)

df.info()

Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 24 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   TransactionID               int64  
 1   DayNumber                   int64  
 2   HourOfSimulation            int64  
 3   SourceAccountID             object 
 4   DestinationAccountID        object 
 5   IsFraud                     int64  
 6   IsFlaggedFraud              int64  
 7   Amount                      float64
 8   OldBalanceOrig              float64
 9   NewBalanceOrig              float64
 10  OldBalanceDest              float64
 11  NewBalanceDest              float64
 12  OriginBalanceChange         float64
 13  DestinationBalanceChange    float64
 14  AmountToOriginBalanceRatio  float64
 15  HighValueTransaction        int64  
 16  BalanceGap                  float64
 17  PeriodOfDay_Evening         int64  
 18  PeriodOfDay_Morning         int64  
 19  P

In [5]:
print("Missing Values Per Column")

df.isnull().sum()

Missing Values Per Column


TransactionID                 0
DayNumber                     0
HourOfSimulation              0
SourceAccountID               0
DestinationAccountID          0
IsFraud                       0
IsFlaggedFraud                0
Amount                        0
OldBalanceOrig                0
NewBalanceOrig                0
OldBalanceDest                0
NewBalanceDest                0
OriginBalanceChange           0
DestinationBalanceChange      0
AmountToOriginBalanceRatio    0
HighValueTransaction          0
BalanceGap                    0
PeriodOfDay_Evening           0
PeriodOfDay_Morning           0
PeriodOfDay_Night             0
TransactionType_CASH_OUT      0
TransactionType_DEBIT         0
TransactionType_PAYMENT       0
TransactionType_TRANSFER      0
dtype: int64

In [6]:
# Target variable
y = df["IsFraud"]

# Features
X = df.drop(columns=[
    "TransactionID",
    "IsFraud",
    "SourceAccountID",
    "DestinationAccountID"
])

print("Feature Matrix Shape :", X.shape)
print("Target Shape :", y.shape)

Feature Matrix Shape : (6362620, 20)
Target Shape : (6362620,)


In [9]:
# Split data BEFORE scaling and SMOTE to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training instances : {X_train.shape[0]:,}")
print(f"Testing instances  : {X_test.shape[0]:,}")

Training instances : 5,090,096
Testing instances  : 1,272,524


In [10]:
# Standardize numerical features
scaler = StandardScaler()

# Fit on training data, transform both (retaining DataFrame structure)
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print("Feature Scaling Completed.")

Feature Scaling Completed.


In [11]:
print("Before SMOTE:")
print(y_train.value_counts())

# Apply SMOTE to the SCALED training data ONLY
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Before SMOTE:
IsFraud
0    5083526
1       6570
Name: count, dtype: int64

After SMOTE:
IsFraud
0    5083526
1    5083526
Name: count, dtype: int64


In [12]:
print("--- Final Shape Verification ---")
print("X_train_smote :", X_train_smote.shape)
print("y_train_smote :", y_train_smote.shape)
print("X_test_scaled :", X_test_scaled.shape)
print("y_test        :", y_test.shape)

print("\n--- Missing Value Verification ---")
print("Missing values in X_train_smote :", X_train_smote.isnull().sum().sum())
print("Missing values in X_test_scaled :", X_test_scaled.isnull().sum().sum())

--- Final Shape Verification ---
X_train_smote : (10167052, 20)
y_train_smote : (10167052,)
X_test_scaled : (1272524, 20)
y_test        : (1272524,)

--- Missing Value Verification ---
Missing values in X_train_smote : 0
Missing values in X_test_scaled : 0


In [13]:
print("Exporting datasets for model training...")

X_train_smote.to_csv("X_train_smote.csv", index=False)
X_test_scaled.to_csv("X_test.csv", index=False)
y_train_smote.to_csv("y_train_smote.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Datasets Exported Successfully")

Exporting datasets for model training...
Datasets Exported Successfully


In [14]:
print("="*60)
print("Enterprise FinTech Payment Intelligence Platform")
print("Notebook 04 - Data Preprocessing Completed")
print("="*60)
print(f"Original Dataset          : {df.shape}")
print(f"Initial Training Dataset  : {X_train.shape}")
print(f"Testing Dataset           : {X_test_scaled.shape}")
print(f"Balanced Training Dataset : {X_train_smote.shape}")
print("\nPreprocessing Completed Successfully")

Enterprise FinTech Payment Intelligence Platform
Notebook 04 - Data Preprocessing Completed
Original Dataset          : (6362620, 24)
Initial Training Dataset  : (5090096, 20)
Testing Dataset           : (1272524, 20)
Balanced Training Dataset : (10167052, 20)

Preprocessing Completed Successfully


In [15]:
print("="*60)
print("Class Distribution Before SMOTE")
print("="*60)
print(y_train.value_counts())

print()

print("="*60)
print("Class Distribution After SMOTE")
print("="*60)
print(y_train_smote.value_counts())

Class Distribution Before SMOTE
IsFraud
0    5083526
1       6570
Name: count, dtype: int64

Class Distribution After SMOTE
IsFraud
0    5083526
1    5083526
Name: count, dtype: int64


In [ ]:
### 💡 Final Business Summary

* Dataset successfully split into training and testing datasets (80/20 split).
* Numerical features standardized using `StandardScaler` to ensure large monetary values do not artificially skew the model.
* Severe fraud class imbalance (0.13%) resolved using **SMOTE** (Synthetic Minority Over-sampling Technique) exclusively on the training set.
* Testing dataset remained untouched by SMOTE to ensure unbiased, real-world model evaluation.
* Final analytical datasets exported and ready for machine learning model training.